# Calculate ETCCDI climate indices

This code calculates the climate indices for any experiments

You will need:
- daily maximum temperature (TREFHTMX)
- daily minimum temperature (TREFHTMN)
- daily precipitation rate (PRECT)

This code generates the following output (see tables below) in the specified folder. The indices calculated here do not include percentile-based indices.

SOURCE: https://xclim.readthedocs.io/en/stable/indices.html#xclim.indices.prcptot_wetdry_period

In [12]:
import xarray as xr
import numpy as np
from os.path import join, expanduser
import xclim.indices
import os
import tempfile
from pathlib import Path
import cftime
import shutil

## First calculate fixed threshold and index indices

### Temperature indices

| Variable Used | Index | Name | Definition | Unit | Type |
|---|---|---|---|---|---|
| TREFHTMX (Tx) | SU | Summer days | Number of days when Tx $>$ 25$^\circ$C | days per year | Fixed threshold |
| TREFHTMX (Tx) | ID | Ice days | Number of days when Tx $<$ 0$^\circ$C | days per year | Fixed threshold |
| TREFHTMX (Tx) | TXn | Coldest days | Annual minimum daily maximum temperature | $^\circ$C | Fixed Index |
| TREFHTMX (Tx) | TXx | Warmest days | Annual maximum daily maximum temperature | $^\circ$C | Fixed Index |
| TREFHTMN (Tn) | TNn | Coldest night | Annual minimum daily minimum temperature | $^\circ$C | Fixed Index |
| TREFHTMN (Tn) | TNx | Warmest night | Annual maximum daily minimum temperature | days per year | Fixed Index |
| TREFHTMN (Tn) | FD | Frost days | Number of days when Tn < 0$^\circ$C | days per year | Fixed threshold |
| TREFHTMN (Tn) | TN | Tropical nights | Number of days when Tn $>$ 20$^\circ$C | days per year | Fixed threshold |

In [ ]:
import dask
num_ensembles = 3 # total number of ensembles

for i in range(1, num_ensembles + 1):
    ensemble_id = f'{(i):03d}'  # 001, 002, 003
    print(f'Loading ensemble member {ensemble_id}')
    file_path = f'/glade/campaign/cesm/collections/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.{ensemble_id}/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.{ensemble_id}.cam.h1.TREFHTMX.*.nc'
    tasmax = xr.open_mfdataset(file_path, parallel=True, combine='nested', concat_dim='time')
    tasmax = tasmax.sel(time=~tasmax.get_index("time").duplicated()) # this removes any duplicates
    tasmax = tasmax.isel(time=slice(None, -1))

    file_path = f'/glade/campaign/cesm/collections/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.{ensemble_id}/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.{ensemble_id}.cam.h1.TREFHTMN.*.nc'
    tasmin = xr.open_mfdataset(file_path, parallel=True, combine='nested', concat_dim='time')
    tasmin = tasmin.sel(time=~tasmin.get_index("time").duplicated())
    tasmin = tasmin.isel(time=slice(None, -1))

    # =======================================================
    # Yearly indices
    # =======================================================

    # Number of days when Tx >= 25 (SU)
    SU = xclim.indices.tx_days_above(tasmax.TREFHTMX, thresh='25.0 degC', freq='YS')
    
    # Number of ice/freezing days (ID)
    ID = xclim.indices.ice_days(tasmax.TREFHTMX, thresh='0 degC', freq='YS')
    
    # Annual Maximum Daily Maximum/Hottest Day (TXX)
    TXX = xclim.indices.tx_max(tasmax.TREFHTMX, freq="YS")

    # Annual Minimum Daily Maximum/Coldest Day (TXN)
    TXN = xclim.indices.tx_min(tasmax.TREFHTMX, freq="YS")
    
    # Annual count of frost days (TN < 0°C)
    FD = xclim.indices.frost_days(tasmin.TREFHTMN, freq='YS')

    # Annual count of tropical nights (TN > 20°C)
    TN = xclim.indices.tn_days_above(tasmin.TREFHTMN, thresh='20.0 degC', freq='YS')

    # Monthly maximum of daily minimum temperature (TNX: warmest night)
    TNX = xclim.indices.tn_max(tasmin.TREFHTMN, freq = 'YS')
    
    # Monthly minimum of daily minimum temperature (TNN: coldest night)
    TNN = xclim.indices.tn_min(tasmin.TREFHTMN, freq = 'YS')

    su, id_, txx, txn, fd, tn, tnx, tnn = dask.compute(SU, ID, TXX, TXN, FD, TN, TNX, TNN)
        
    #path to save ETCCDI data
    sunm = join(f'/path_to_save_ETCCDI/', 'SU.nc')
    idnm  = join(f'/path_to_save_ETCCDI/', 'ID.nc')
    txxnm = join(f'/path_to_save_ETCCDI/', 'TXX.nc')
    txnnm = join(f'/path_to_save_ETCCDI/', 'TXN.nc')
    fdnm = join(f'/path_to_save_ETCCDI/', 'FD.nc')
    tnnm = join(f'/path_to_save_ETCCDI/', 'TN.nc')
    tnxnm = join(f'/path_to_save_ETCCDI/', 'TNX.nc')
    tnnnm = join(f'/path_to_save_ETCCDI/', 'TNN.nc')

    su.to_netcdf(sunm)
    id_.to_netcdf(idnm)
    txx.to_netcdf(txxnm)
    txn.to_netcdf(txnnm)
    fd.to_netcdf(fdnm)
    tn.to_netcdf(tnnm)
    tnx.to_netcdf(tnxnm)
    tnn.to_netcdf(tnnnm)


Loading ensemble member 001
Loading ensemble member 002
Loading ensemble member 003


### Precipitation indices

In [7]:
# To convert precipitation rate to mm/day
SECONDS_PER_DAY = 86400
M_TO_MM = 1000

def to_mm_day(da, varname=None):
    """
    Convert precipitation rate to mm/day when units look like:
      - kg m-2 s-1  (water flux; 1 kg/m^2 = 1 mm water)
      - m s-1       (water-equivalent depth rate)
    Otherwise return da unchanged.
    """
    units = (da.attrs.get("units") or "").strip().lower().replace(" ", "")

    # Common unit spellings
    is_kg_m2_s = units in {"kgm-2s-1", "kg/m^2/s", "kg/m2/s","kgm^-2s^-1", "kgm**-2s**-1"}
    is_m_s     = units in {"ms-1", "m/s", "m s-1", "m/s-1"} or units == "ms^-1"

    if is_kg_m2_s:
        out = da * SECONDS_PER_DAY  # kg/m^2/day == mm/day
    elif is_m_s:
        out = da * SECONDS_PER_DAY * M_TO_MM
    else:
        # Fallback: if this is PRECT/precip by name, assume kg m-2 s-1 (CESM PRECT convention)
        if varname is not None and varname.upper() in {"PRECT", "PRECC", "PRECL", "PR"}:
            out = da * SECONDS_PER_DAY
        else:
            return da

    out = out.assign_attrs(da.attrs)
    out.attrs["units"] = "mm/day"
    return out

| Variable Used | Index | Name | Definition | Unit | Type |
|---|---|---|---|---|---|
| PRECT (PR) | PRCPTOT | Total rainfall | Annual sum of precipitation | mm | Fixed Index |
| PRECT (PR) | SDII | Smple daily intensity | Mean precipitation falling on days when PR $\geq$ 1 mm | mm | Fixed Index |
| PRECT (PR) | Rx1day | Wettest day | Annual maximum precipitation in a single day | mm | Fixed Index |
| PRECT (PR) | Rx5day | Wettest pentad | Annual maximum precipitation falling on 5 consecutive days | mm | Fixed Index |
| PRECT (PR) | CDD | Consecutive dry days | Longest spell of consecutive days when PR $\leq$ 1 mm | days per year | Fixed index/spell |
| PRECT (PR) | CWD | Consecutive wet days | Longest spell of consecutive days when PR $\geq$ 1 mm | days per year | Fixed index/spell |
| PRECT (PR) | R10mm | Heavy precipitation days | Number of days when precipitation $\geq$ 10 mm | days per year | Fixed threshold |
| PRECT (PR) | R20mm | Very heavy precipitation days | Number of days when precipitation $\geq$ 20 mm | days per year | Fixed threshold |

In [ ]:
import dask
num_ensembles = 3 # total number of ensembles

for i in range(1, num_ensembles + 1):
    ensemble_id = f'{(i):03d}'  # 001, 002, 003
    print(f'Loading ensemble member {ensemble_id}')
    file_path = f'/glade/campaign/cesm/collections/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.{ensemble_id}/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.{ensemble_id}.cam.h1.PRECT.*.nc' 
    ds = xr.open_mfdataset(file_path, parallel=True, combine='nested', concat_dim='time')
    ds = ds.sel(time=~ds.get_index("time").duplicated())
    ds = ds.isel(time=slice(None, -1))
    pr = to_mm_day(ds["PRECT"], varname="PRECT") #convert to mm/day here

    # =======================================================
    # Yearly indices
    # =======================================================

    # Monthly total precipitation (PRCPTOT)
    PRCPTOT = xclim.indices.prcptot(pr, freq='YS')

    # Simple daily intensity index (SDII)
    SDII = xclim.indices.daily_pr_intensity(pr, thresh='1 mm/day', freq='YS')

    # Annual maximum 1-day precipitation (RX1D)
    RX1D = xclim.indices.max_1day_precipitation_amount(pr, freq='YS')
    
    # Annual maximum 5-day precipitation (RX5D)
    RX5D = xclim.indices.max_n_day_precipitation_amount(pr, window=5, freq='YS')

    # Max consecutive dry days per year (CDD)
    CDD = xclim.indices.maximum_consecutive_dry_days(pr, '1 mm/day', freq='YS')

    # Max consecutive wet days per year (CWD)    
    CWD = xclim.indices.maximum_consecutive_wet_days(pr, thresh='1 mm/day', freq='YS')

    # Annual number of days with precip ≥ 10 mm (R10mm)
    R10MM = xclim.indices.wetdays(pr, thresh='10 mm/day', freq='YS', op='>=')

    # Monthly number of days with precip ≥ 20 mm (R20mm)
    R20MM = xclim.indices.wetdays(pr, thresh='20 mm/day', freq='YS', op='>=') 

    prcptot, sdii, rx10, rx5d, cdd, cwd, r10mm, r20mm = dask.compute(PRCPTOT, SDII, RX1D, RX5D, CDD, CWD, R10MM, R20MM)
    
    #path to save ETCCDI data
    prcptotm = join(f'/path_to_save_ETCCDI/', 'PRCPTOT.nc')
    sdiinm  = join(f'/path_to_save_ETCCDI/', 'SDII.nc')
    rx1dnm = join(f'/path_to_save_ETCCDI/', 'RX1D.nc')
    rx5dnm = join(f'/path_to_save_ETCCDI/', 'RX5D.nc')
    cddnm = join(f'/path_to_save_ETCCDI/', 'CDD.nc')
    cwdnm = join(f'/path_to_save_ETCCDI/', 'CWD.nc')
    r10mmnm = join(f'/path_to_save_ETCCDI/', 'R10MM.nc')
    r20mmnm = join(f'/path_to_save_ETCCDI/', 'R20MM.nc')

    prcptot.to_netcdf(prcptotm)
    sdii.to_netcdf(sdiinm)
    rx10.to_netcdf(rx1dnm)
    rx5d.to_netcdf(rx5dnm)
    cdd.to_netcdf(cddnm)
    cwd.to_netcdf(cwdnm)
    r10mm.to_netcdf(r10mmnm)
    r20mm.to_netcdf(r20mmnm)

